# [Step 3] 관성 상승 구간의 아포지 회귀: 현재 r 정의를 유지하는 출력 구조

이 실습에서는 추력이 없는 관성 상승 구간만 사용합니다. 목표는 **r = H / (h + v_z²/(2g))** 그대로입니다.
일반 ANN과 제약 ANN의 출력 구조만 비교합니다. 학습 성능이나 실기 정확도는 아직 검증하지 않았습니다.
기존 실행 출력과 성과 수치는 모두 제거했습니다.


## 0.1 범위와 입력 상태

`flight_phase == "coasting"`, v_z > 3 m/s인 행을 대상으로 합니다. h는 현재 AGL 고도이고 H는 같은 발사 기준의 아포지입니다.
시뮬레이션에서는 head의 h와 v_z에 깨끗한 상태를 사용합니다. 이는 UKF 성능을 검증한 실험이 아닙니다.
실기 로그의 필터 상태와 같은 품질이라고 가정하지 않습니다. 연소 구간, 저속 구간, 하강 구간의 성능도 주장하지 않습니다.


## 0.2 바꾸지 않는 목표와 좌표 기준

G = v_z²/(2g), D = h + G, r = H/D, H = D·r로 정의합니다.
기존 ASL `energy_ratio`는 보존하고, metadata의 `apogee_altitude`에서 feature 전처리와 **동일한 발사 고도**를 빼서 `apogee_agl`을 만듭니다.
새 학습 라벨은 `r_target`입니다. 좌표 원점을 바꾸면 r도 달라지므로 기존 ASL 모델이나 scaler를 재사용하지 않습니다.
남은 상승량의 비율로 목표를 바꾸지 않습니다.


## 1.1 탄도 상한의 조건

무추력 상승에서 수직 비중력 가속도 a_ng,z의 일이 양수가 아니라고 가정합니다.
수직 운동식 v_z·dv_z/dh = −g + a_ng,z를 현재 h에서 아포지 H까지 적분하면 다음과 같습니다.

v_z²/2 = g(H − h) − ∫ a_ng,z dh ≥ g(H − h)

따라서 h ≤ H ≤ h + G이며 h/D ≤ r ≤ 1입니다.
전체 3차원 에너지가 소산된다는 조건만으로 이 수직 상한이 자동으로 성립하지는 않습니다.
바람, 양력, 상태 추정 오차가 있는 데이터는 이 단순 모델의 경계를 넘을 수 있으므로 라벨을 직접 검사합니다.


## 1.2 상태에 따라 달라지는 하한

h ≥ 0, v_z > 3, 유한한 D > 0에서 r_floor = h/D로 둡니다.
같은 신경망이 만든 유한한 logit에 대해 다음 head를 사용합니다.

**r = r_floor + (1 − r_floor)·sigmoid(logit)**

**H = D·r**

이 구조는 부동소수점 오차 범위에서 h ≤ H ≤ D와 r ≤ 1을 만족합니다.
sigmoid의 값은 내부 출력 매개화이며 별도의 정답 라벨이 아닙니다. 학습은 r와 H의 지도 오차로 수행합니다.


## 1.3 일반 ANN과 제약 ANN의 차이

두 모델은 같은 6개 feature, 같은 64·32 은닉층, 같은 Tanh 활성화와 같은 파라미터 수를 사용합니다.
초기 가중치를 복제하고 비행 분할, 행, seed, 배치 순서, optimizer, epoch 예산과 손실을 동일하게 맞춥니다.
일반 ANN은 마지막 선형 출력 자체를 r로 사용하고, 제약 ANN만 상태 의존 head를 적용합니다.
일반 ANN의 결과에는 사후 보정을 하지 않습니다. 이 비교는 출력 구조 강제의 실습이며 PDE 해를 구하는 PINN 실험이 아닙니다.


## 1.4 보장하는 것과 보장하지 않는 것

보장 대상은 지정한 유효 입력에서의 출력 구간입니다. 잘못된 입력은 작은 분모로 대체하지 않고 거부합니다.
추진 중에도 같은 경계가 맞는다고 해석하지 않습니다. 운동방정식 전체, 외삽 정확도, 학습 성공이나 운용 안전성을 보장하지 않습니다.
현재 고도를 그대로 출력하는 하한 기준선과 r=1 탄도 기준선을 학습 모델과 별도로 보고합니다.


## 1.5 라벨 감사와 엄격한 부분집합

후보는 관성 상승, 유한한 feature와 상태, h ≥ 0, D > 0, 유한한 H와 r 조건으로 정합니다.
그다음 정답에 대해 h ≤ H ≤ D를 **허용 오차 없이** 검사합니다. r > 1인 작은 위반도 잘라내지 않습니다.
학습과 검증에서는 위반 행을 제외하고 제외 수와 이유를 공개합니다.
누락된 metadata, 누락 라벨, 비유한 라벨, 입력 누락, 비유한 입력과 경계 위반은 구별해서 집계합니다.


## 1.6 정답 선별 없는 주 시험

주 시험 지표는 **모든 유효 관성 상승 시험 후보**에서 계산합니다. 정답이 경계를 넘더라도 포함합니다.
정답 경계가 맞는 `test_strict`는 보조 지표일 뿐입니다. 정답이 없거나 비유한 행은 채점할 수 없으므로 별도 누락 수를 보고합니다.
각 보고에는 행 수와 비행 수를 포함하고 r MAE·RMSE, H MAE·RMSE, 비행별 H MAE의 평균을 함께 계산합니다.
행 평균은 긴 비행의 영향을 더 받으므로 비행별 평균과 구별합니다.


## 1.7 불필요한 물리 손실은 사용하지 않습니다

head가 이미 강제하는 범위에 다시 벌점을 주지 않습니다. 연소 여부에 따라 켜고 끄는 손실도 없습니다.
시간 좌표의 편미분을 실제 궤적의 단조성으로 해석하지 않으며, 시간 편미분 손실을 사용하지 않습니다.
자동 미분은 두 모델의 동일한 지도 손실을 가중치로 역전파할 때만 사용합니다.


## 2.1 전처리와 독립 분할

공유 `apogee_pipeline.make_sim_features`를 사용하여 AGL 고도, 중력 제거 가속도, 동압 proxy를 만듭니다.
feature 순서는 공유 `FEATURE_COLUMNS` 그대로이며 시간은 입력하지 않습니다.
동압은 시뮬레이션의 깨끗한 h와 v_z를 쓰는 발사 기준 대기밀도 proxy입니다. 실제 대기밀도 측정값은 아닙니다.
공유 비행 분할은 200편일 때 train/validation/calibration/test = 120/30/25/25입니다.
scaler는 선별된 train 행에서만 학습합니다. calibration은 모델 선택이나 early stopping에 사용하지 않습니다.


In [ ]:
from pathlib import Path
import copy
import json
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import apogee_pipeline as ap
import coasting_apogee as coast

SEED = 42
RUN_TRAINING = False  # 명시적으로 켜기 전에는 torch를 import하지 않습니다.
SAVE_ARTIFACTS = False
DATA_PATH = Path("data/simulated/all_trajectories.csv.gz")
META_PATH = Path("data/simulated/flight_metadata.csv.gz")
OUTPUT_DIR = Path("data/experiments/step3-coasting")
REAL_PATH = Path("data/112.txt")
REAL_LAUNCH_TIME_S = None  # 로그 시스템 시각을 외부 확인 후 설정합니다.
REAL_COAST_START_S = None  # 발사 후 초 단위입니다. 실기 연소 종료 확인 전에는 비워 둡니다.
EPOCHS = 100
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
H_SCALE_M = 100.0
H_LOSS_WEIGHT = 0.1
np.random.seed(SEED)
raw_df = pd.read_csv(DATA_PATH)
metadata_df = pd.read_csv(META_PATH)
print({"rows": len(raw_df), "flights": raw_df.flight_id.nunique(), "training_enabled": RUN_TRAINING})


In [ ]:
# 원본 energy_ratio를 덮어쓰지 않습니다. flight_phase가 없으면 명시적으로 실패합니다.
df = coast.make_coasting_frame(raw_df, metadata_df)
feature_cols = list(coast.FEATURE_COLUMNS)
if "energy_ratio" in raw_df:
    np.testing.assert_array_equal(df["energy_ratio"], raw_df["energy_ratio"])
print(df["input_rejection"].value_counts().to_string())
print(df["label_status"].value_counts().to_string())


In [ ]:
split = ap.build_split_manifest(df["flight_id"].unique(), seed=SEED)
parts, audit = coast.partition_coasting(df, split)
train_df = parts["train"]
val_df = parts["validation"]
cal_df = parts["calibration"]
test_df = parts["test"]
test_strict_df = parts["test_strict"]
print("비행 수:", {name: len(ids) for name, ids in split.items()})
print("행 감사: 입력 거부 이유는 상호 배타적이며 라벨 상태는 별도로 집계합니다.")
print(audit.to_string(index=False))
if train_df.empty or val_df.empty or test_df.empty:
    raise ap.PipelineError("train, validation and full test need nonempty valid populations")


## 2.4 학습 배열의 계약

정규화 feature와 물리 단위 h, v_z, r_target, apogee_agl을 분리합니다.
scaler는 train에만 fit하고 나머지는 transform만 합니다. 일반 ANN과 제약 ANN이 같은 배열을 공유합니다.
float32 신경망 입력과 float64 물리 상태·라벨을 사용합니다. head는 물리 상태의 dtype에서 계산합니다.


In [ ]:
feature_scaler = StandardScaler()
feature_scaler.fit(train_df[feature_cols])

def arrays_for(frame):
    x = (feature_scaler.transform(frame[feature_cols]).astype(np.float32)
         if len(frame) else np.empty((0, len(feature_cols)), dtype=np.float32))
    return {
        "x": x,
        "h": frame["altitude_rel"].to_numpy(dtype=np.float64),
        "v": frame["velocity_z"].to_numpy(dtype=np.float64),
        "r": frame["r_target"].to_numpy(dtype=np.float64),
        "H": frame["apogee_agl"].to_numpy(dtype=np.float64),
    }

arrays = {name: arrays_for(frame) for name, frame in parts.items()}
np.testing.assert_allclose(feature_scaler.mean_, train_df[feature_cols].mean().to_numpy())


In [ ]:
def build_models(input_dim, seed):
    # 실행 요청이 있을 때만 torch를 불러옵니다. 여기서는 소스만 제공합니다.
    import torch
    from torch import nn

    torch.manual_seed(seed)
    base = nn.Sequential(
        nn.Linear(input_dim, 64), nn.Tanh(),
        nn.Linear(64, 32), nn.Tanh(), nn.Linear(32, 1),
    )

    class BaselineANN(nn.Module):
        def __init__(self, network):
            super().__init__()
            self.network = network

        def forward(self, x, h, v):
            if (x.ndim != 2 or h.ndim != 1 or v.shape != h.shape
                    or len(x) != len(h)):
                raise ValueError("incompatible feature/state shapes")
            denom = h + v.square() / (2.0 * ap.SIM_G)
            valid = (torch.isfinite(x).all() & torch.isfinite(h).all()
                     & torch.isfinite(v).all() & torch.isfinite(denom).all()
                     & (h >= 0).all() & (v > 3).all() & (denom > 0).all())
            if not bool(valid):
                raise ValueError("requires finite coasting h >= 0, v > 3, D > 0")
            logit = self.network(x).squeeze(-1).to(h.dtype)
            if not bool(torch.isfinite(logit).all()):
                raise ValueError("nonfinite network output")
            return logit, denom * logit

    class ConstrainedANN(BaselineANN):
        def forward(self, x, h, v):
            logit, _ = super().forward(x, h, v)
            denom = h + v.square() / (2.0 * ap.SIM_G)
            r_floor = h / denom
            ratio = r_floor + (1.0 - r_floor) * torch.sigmoid(logit)
            return ratio, denom * ratio

    return {
        "baseline_ann": BaselineANN(copy.deepcopy(base)),
        "constrained_ann": ConstrainedANN(copy.deepcopy(base)),
    }


In [ ]:
def supervised_loss(r_pred, H_pred, r_target, H_target):
    # 두 모델에 동일한 무차원 지도 손실을 사용합니다.
    return ((r_pred - r_target).square().mean()
            + H_LOSS_WEIGHT * ((H_pred - H_target) / H_SCALE_M).square().mean())


In [ ]:
# 유효 영역, 역변환, 출력 경계의 NumPy 확인입니다. 학습 성능 시험은 아닙니다.
def check_numpy_contract(frame):
    restored = coast.reconstruct_apogee(frame["r_target"], frame["altitude_rel"], frame["velocity_z"])
    np.testing.assert_allclose(restored, frame["apogee_agl"], rtol=1e-12, atol=1e-10)
    for logit in (-1000.0, 0.0, 1000.0):
        ratio, height = coast.constrained_r_head(logit, frame["altitude_rel"], frame["velocity_z"])
        assert np.all(ratio <= 1.0)
        assert np.all(height >= frame["altitude_rel"].to_numpy() - 1e-10)
        assert np.all(height <= frame["denom"].to_numpy() + 1e-10)

check_numpy_contract(test_df)


In [ ]:
def tensor_arrays(data, device):
    import torch
    return {name: torch.as_tensor(value, device=device) for name, value in data.items()}


## 4.3 동일한 학습 예산과 최선 상태 복원

두 모델을 각각 같은 100 epoch로 학습하도록 작성했습니다. 배치 순서는 각 학습 시작 시 같은 seed로 재설정합니다.
검증 지도 손실이 가장 작은 state_dict를 deepcopy하고, 학습 종료 후 그 상태를 복원합니다.
조기 종료로 두 모델의 업데이트 수가 달라지지 않게 고정 예산을 사용합니다.
본 수정에서는 torch import, 설치, 학습과 추론을 수행하지 않았습니다. 아래 경로는 실행 검증 전 소스입니다.


In [ ]:
def train_one(model, train_data, validation_data):
    import torch
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    device = torch.device("cpu")
    model = model.to(device)
    train_t = tensor_arrays(train_data, device)
    val_t = tensor_arrays(validation_data, device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    history = []
    best_state = None
    best_val = float("inf")
    for epoch in range(EPOCHS):
        model.train()
        order = torch.randperm(len(train_t["x"]), device=device)
        loss_sum = 0.0
        for start in range(0, len(order), BATCH_SIZE):
            idx = order[start:start + BATCH_SIZE]
            optimizer.zero_grad()
            r_pred, H_pred = model(train_t["x"][idx], train_t["h"][idx], train_t["v"][idx])
            loss = supervised_loss(r_pred, H_pred, train_t["r"][idx], train_t["H"][idx])
            if not bool(torch.isfinite(loss)):
                raise RuntimeError("nonfinite training loss")
            loss.backward()
            optimizer.step()
            loss_sum += float(loss.detach()) * len(idx)
        model.eval()
        with torch.no_grad():
            r_pred, H_pred = model(val_t["x"], val_t["h"], val_t["v"])
            validation_loss = float(supervised_loss(r_pred, H_pred, val_t["r"], val_t["H"]))
        if not np.isfinite(validation_loss):
            raise RuntimeError("nonfinite validation loss")
        history.append({"epoch": epoch + 1, "train_loss": loss_sum / len(order),
                        "validation_loss": validation_loss})
        if validation_loss < best_val:
            best_val = validation_loss
            best_state = copy.deepcopy(model.state_dict())
    if best_state is None:
        raise RuntimeError("no finite best checkpoint")
    model.load_state_dict(best_state)
    model.eval()
    return model, history


## 4.4 calibration의 위치

calibration 비행은 독립 보관하며 이번 ANN 비교에서는 사용하지 않습니다.
예측 구간이나 conformal 보장도 계산하지 않습니다. 이후 분석에서 calibration을 쓰려면 점수와 보장 단위를 별도로 설계해야 합니다.
시험 비행을 모델 선택이나 scaler 학습에 사용하지 않습니다.


In [ ]:
models = {}
histories = {}
trained = False
if RUN_TRAINING:
    models = build_models(len(feature_cols), SEED)
    initial_states = [model.state_dict() for model in models.values()]
    for key in initial_states[0]:
        np.testing.assert_array_equal(initial_states[0][key].numpy(), initial_states[1][key].numpy())
    for name, model in models.items():
        models[name], histories[name] = train_one(model, arrays["train"], arrays["validation"])
    trained = True
else:
    print("skip: RUN_TRAINING=False; torch와 학습 모델을 실행하지 않았습니다.")


In [ ]:
def predict_ratios(model, data):
    import torch
    if len(data["x"]) == 0:
        return np.empty(0, dtype=float)
    device = next(model.parameters()).device
    values = tensor_arrays(data, device)
    predictions = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(values["x"]), BATCH_SIZE):
            stop = start + BATCH_SIZE
            ratio, _ = model(values["x"][start:stop], values["h"][start:stop], values["v"][start:stop])
            predictions.append(ratio.cpu().numpy())
    return np.concatenate(predictions)


In [ ]:
learned_metrics = {}
if trained:
    for name, model in models.items():
        learned_metrics[name] = {}
        for population in ("test", "test_strict"):
            ratio = predict_ratios(model, arrays[population])
            learned_metrics[name][population] = coast.score_predictions(parts[population], ratio)
    print("주 지표는 test이며 test_strict는 정답 경계가 맞는 보조 부분집합입니다.")
    print(json.dumps(learned_metrics, indent=2, ensure_ascii=False))
else:
    print("skip: 학습 모델 지표는 없습니다. 주 시험은 정답 경계로 선별하지 않습니다.")


In [ ]:
if trained:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for name, history in histories.items():
        table = pd.DataFrame(history)
        axes[0].plot(table["epoch"], table["train_loss"], label=name)
        axes[1].plot(table["epoch"], table["validation_loss"], label=name)
    axes[0].set_title("Training supervised loss")
    axes[1].set_title("Validation supervised loss")
    for axis in axes:
        axis.set_xlabel("Epoch")
        axis.legend()
    fig.tight_layout()
    plt.show()


## 5.3 신경망 없는 기준선을 별도로 보고합니다

하한 기준선은 r = h/D, H = h입니다. constant-r 기준선은 r = 1, H = D입니다.
이는 학습하지 않은 결정론적 기준선이며 학습 모델 성능으로 해석하지 않습니다.
저속이나 끝 시점에서 현재 고도가 정답에 가까워지는 현상을 학습 성공으로 주장하지 않습니다.
이번 성능 모집단은 v_z > 3인 관성 상승 행으로 제한됩니다.


In [ ]:
reference_metrics = {}
for population in ("test", "test_strict"):
    frame = parts[population]
    reference_metrics[population] = {
        "floor_only_not_learned": coast.score_predictions(frame, (frame["altitude_rel"] / frame["denom"]).to_numpy()),
        "constant_r_one_not_learned": coast.score_predictions(frame, np.ones(len(frame))),
    }
print("학습하지 않은 기준선입니다. ANN 성능은 아닙니다.")
print(json.dumps(reference_metrics, indent=2, ensure_ascii=False))


In [ ]:
# 기본값에서는 디렉터리도 만들지 않습니다. 기존 data/model 저장물은 사용하지 않습니다.
if SAVE_ARTIFACTS:
    if not trained:
        raise RuntimeError("refusing artifact write before explicit training")
    import joblib
    import torch
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for name, model in models.items():
        torch.save(model.state_dict(), OUTPUT_DIR / f"{name}_best.pt")
    joblib.dump(feature_scaler, OUTPUT_DIR / "feature_scaler.pkl")
    audit.to_csv(OUTPUT_DIR / "row_audit.csv", index=False)
    manifest = {
        "target": "r=H/(h+v_z^2/(2g)); H and h AGL", "features": feature_cols,
        "feature_units": ap.FEATURE_UNITS, "state_source": "simulation_clean",
        "split": split, "seed": SEED, "epochs": EPOCHS, "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
        "hidden_dims": [64, 32], "hidden_activation": "Tanh",
        "heads": {"baseline_ann": "linear r", "constrained_ann": "h/D+(1-h/D)*sigmoid(logit)"},
        "loss": {"r_mse_weight": 1.0, "H_mse_weight": H_LOSS_WEIGHT, "H_scale_m": H_SCALE_M},
        "data": str(DATA_PATH), "metadata": str(META_PATH),
        "data_sha256": ap.sha256_file(DATA_PATH), "metadata_sha256": ap.sha256_file(META_PATH),
        "primary_population": "all finite-label input-valid ascending coasting test rows",
        "secondary_population": "test rows satisfying h<=H<=D",
        "learned_metrics": learned_metrics, "nonlearned_reference_metrics": reference_metrics,
        "histories": histories,
    }
    ap.write_json(OUTPUT_DIR / "experiment.json", manifest)
else:
    print("skip: SAVE_ARTIFACTS=False; 새 산출물을 생성하지 않았습니다.")


In [ ]:
# 실기 연소 종료 시각을 시뮬레이션 설정에서 가져오지 않습니다.
real_df, real_status = coast.load_real_coasting(
    REAL_PATH, launch_time_s=REAL_LAUNCH_TIME_S, coast_start_s=REAL_COAST_START_S,
)
print(real_status)
# 독립 아포지 참조가 없으므로 실기 오차·MAE·RMSE를 만들지 않습니다.


In [ ]:
if real_df is not None and trained:
    import matplotlib.pyplot as plt
    real_data = {
        "x": feature_scaler.transform(real_df[feature_cols]).astype(np.float32),
        "h": real_df["altitude_rel"].to_numpy(dtype=np.float64),
        "v": real_df["velocity_z"].to_numpy(dtype=np.float64),
    }
    ratio = predict_ratios(models["constrained_ann"], real_data)
    prediction = coast.reconstruct_apogee(ratio, real_data["h"], real_data["v"])
    ballistic = real_data["h"] + real_data["v"]**2 / (2 * ap.SIM_G)
    fig, axis = plt.subplots(figsize=(10, 4))
    axis.plot(real_df["time"], real_data["h"], label="Current measured/filtered h AGL")
    axis.plot(real_df["time"], ballistic, label="Ballistic D (not learned)")
    axis.plot(real_df["time"], prediction, label="Constrained ANN forecast")
    axis.axhline(real_df.attrs["observed_max_agl"], linestyle=":",
                 label="Observed log maximum (not independent truth)")
    axis.set_xlabel("Time after configured launch (s)")
    axis.set_ylabel("AGL altitude (m)")
    axis.legend()
    fig.tight_layout()
    plt.show()
else:
    print("skip: 실기 관성 시작 설정과 새 학습 모델이 모두 있어야 진단 그림을 그립니다.")


## 7.1 해석과 남은 검증

이 수정에서 확인한 대상은 NumPy/pandas 전처리, 라벨 정의의 역변환, 유효 영역 거부, 공유 분할과 출력 구조의 NumPy 참조입니다.
노트북의 torch 경로는 소스만 작성했으며 학습·추론·성능을 실행 검증하지 않았습니다.
출력 경계가 맞는다는 사실은 예측 오차가 작다는 증거가 아닙니다. 정답 경계 밖 행을 포함하는 주 시험 결과와 감사 수를 함께 해석해야 합니다.

실기는 `REAL_COAST_START_S=None`이므로 기본적으로 명시적으로 건너뜁니다. 확인된 실기 연소 종료 정보가 필요합니다.
시작 시각을 설정하더라도 축 매핑, 센서 보정, 시뮬레이션과 실기의 상태 차이는 별도로 확인해야 합니다.
로그 최대 고도는 관측 요약일 뿐 독립 아포지 정답이 아니므로 실기 예측 오차를 보고하지 않습니다.

산출물은 명시적으로 학습과 저장을 켠 경우에만 `data/experiments/step3-coasting/`에 저장합니다.
기존 Step 4의 고정 GRU, 다른 노트북, 기존 scaler와 모델 파일은 변경하지 않습니다.
